# Kaggle setup
Thin wrappers around repository CLIs. Enable Internet for installation and select a GPU accelerator. Source and result archives are separate uploads. No previous working directory is assumed to survive a session.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
INPUT_DIR = Path(os.environ.get("SEQREC_INPUT_DIR", "/kaggle/input"))
WORK_DIR = Path(os.environ.get("SEQREC_WORK_DIR", "/kaggle/working"))
REPO = Path(os.environ.get("SEQREC_REPO_DIR", str(WORK_DIR / "sequential-recommendation")))
PYTHON = os.environ.get("SEQREC_PYTHON", str(WORK_DIR / "seqrec-env/bin/python"))
if not Path(PYTHON).exists():
    PYTHON = sys.executable
def run(*args):
    subprocess.run([PYTHON, *map(str, args)], cwd=REPO, check=True)
print({"repo": str(REPO), "python": PYTHON, "input": str(INPUT_DIR), "work": str(WORK_DIR)})


In [ ]:
# Set these to your attached Kaggle Dataset paths.
SOURCE_ARCHIVE = INPUT_DIR / "seqrec-source/seqrec-kaggle-source.zip"
PRIOR_RESULTS_ARCHIVE = None  # e.g. INPUT_DIR / "previous-run/seqrec-results-session1.zip"
REPO.mkdir(parents=True, exist_ok=True)
# Validate paths before extracting the trusted source package.
import zipfile
with zipfile.ZipFile(SOURCE_ARCHIVE) as archive:
    for member in archive.infolist():
        target = (REPO / member.filename).resolve()
        if not target.is_relative_to(REPO.resolve()):
            raise ValueError("Unsafe archive path")
        if target.exists() and not member.is_dir():
            raise ValueError("Repository already exists: use it or select a fresh REPO directory")
    archive.extractall(REPO)
print(REPO)


In [ ]:
# Use a separate supported Python environment when the Kaggle kernel is newer.
if sys.version_info[:2] not in [(3, 10), (3, 11)]:
    subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
    subprocess.run([sys.executable, "-m", "uv", "python", "install", "3.11"], check=True)
    subprocess.run([sys.executable, "-m", "uv", "venv", "--seed", "--python", "3.11", str(WORK_DIR / "seqrec-env")], check=True)
    PYTHON = str(WORK_DIR / "seqrec-env/bin/python")
run("-m", "pip", "install", "-r", "requirements-lock.txt", "--extra-index-url", "https://download.pytorch.org/whl/cu121")
run("scripts/check_environment.py")
run("scripts/run_smoke_test.py", "--device", "cpu")
# Run CUDA smoke after check_environment confirms a usable accelerator.
run("scripts/run_smoke_test.py", "--device", "cuda")


In [ ]:
if PRIOR_RESULTS_ARCHIVE is not None:
    run("scripts/kaggle_session.py", "restore", "--archive", PRIOR_RESULTS_ARCHIVE, "--destination", REPO)
    subprocess.run([PYTHON, "scripts/verify_results.py", "--root", "results/core"], cwd=REPO, check=False)  # INCOMPLETE is expected before resume
# Existing processed bundles may be attached read-only or packaged with source.
DATASET = "beauty"
BUNDLE = None  # set Path('/kaggle/input/seqrec-data/data/processed/beauty/HASH') to reuse an attached bundle
if BUNDLE is None:
    existing = sorted((REPO / "data/processed" / DATASET).glob("*/manifest.json"))
    if not existing:
        run("scripts/download_data.py", "--dataset", DATASET)
        run("scripts/preprocess_data.py", "--dataset", DATASET)
    bundles = sorted((REPO / "data/processed" / DATASET).glob("*/manifest.json"))
    if len(bundles) != 1:
        raise ValueError("Select the exact bundle explicitly; multiple or no bundles found")
    BUNDLE = bundles[0].parent
run("scripts/verify_data.py", "--bundle", BUNDLE, "--output", "results/reports/kaggle_data_audit.json")
(WORK_DIR / "seqrec_session.json").write_text(json.dumps({"repo": str(REPO), "python": PYTHON, "bundle": str(BUNDLE), "dataset": DATASET}, indent=2))


Save seqrec_session.json and the exported result ZIP as notebook outputs. A later Kaggle session must attach the ZIP and restore it; /kaggle/working is temporary.